### Data Ingestion for BNS 2023

In [1]:
import fitz
import re
import numpy as np
import os

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

from pymongo import MongoClient

c:\Users\Himanshu\Desktop\LexiBNS\.myenv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
c:\Users\Himanshu\Desktop\LexiBNS\.myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
PDF_PATH = r"./THE_BHARATIYA_NYAYA_SANHITA_2023.pdf"
LAW = "BNS"
MAX_SECTION = 358

In [6]:
def extract_main_text(page):

    blocks = page.get_text("blocks")
    kept_blocks = []
    for block in blocks:

        x0, y0, x1, y1, text, *_ = block

        # Central statutory text area
        if (
            x0 >= 110
            and x1 <= 490
            and 80 <= y0 <= 780
        ):

            text = text.strip()

            if not text:
                continue

            upper = text.upper()

            # Remove Gazette headers
            if "THE GAZETTE OF INDIA" in upper:
                continue

            if "[PART II" in upper:
                continue

            kept_blocks.append(
                (y0, x0, text)
            )

    # Preserve reading order
    kept_blocks.sort(
        key=lambda x: (x[0], x[1])
    )

    return "\n".join(
        text
        for _, _, text in kept_blocks
    )

In [7]:
pdf = fitz.open(PDF_PATH)

full_text = ""

for page_number, page in enumerate(pdf,start=1):
    page_text = extract_main_text(page)
    full_text += (
        f"\n<<<PAGE:{page_number}>>>\n"
        + page_text
        + "\n"
    )


print("PDF pages:", len(pdf))
print("Characters extracted:", len(full_text))

PDF pages: 102
Characters extracted: 347254


In [8]:
def parse_bns_sections(
    full_text,
    pdf_path
):

    matches = []

    search_position = 0

    # Search specifically for:
    # 1, then 2, then 3 ... up to 358

    for section_number in range(
        1,
        MAX_SECTION + 1
    ):

        pattern = re.compile(
            rf"(?m)^\s*{section_number}\.\s*"
        )

        match = pattern.search(
            full_text,
            search_position
        )

        if match is None:

            raise ValueError(
                f"Section {section_number} not found"
            )

        matches.append(
            (
                section_number,
                match
            )
        )

        search_position = match.end()

    sections = []

    for i, (
        section_number,
        match
    ) in enumerate(matches):

        start = match.start()

        if i + 1 < len(matches):

            end = matches[
                i + 1
            ][1].start()

        else:

            end = len(full_text)

        raw_section = full_text[
            start:end
        ]

        # Detect source pages
        pages = [
            int(page)
            for page in re.findall(
                r"<<<PAGE:(\d+)>>>",
                raw_section
            )
        ]

        # Remove markers from actual text
        clean_text = re.sub(
            r"<<<PAGE:\d+>>>",
            "",
            raw_section
        ).strip()

        sections.append(
            Document(
                page_content=clean_text,

                metadata={
                    "law": "BNS",

                    "section":
                        str(section_number),

                    "pages":
                        sorted(set(pages)),

                    "source":
                        pdf_path,

                    "document_type":
                        "primary_statute"
                }
            )
        )

    return sections

In [9]:
sections = parse_bns_sections(
    full_text,
    PDF_PATH
)

print(
    "Sections extracted:",
    len(sections)
)

Sections extracted: 358


In [10]:
section_numbers = [
    int(doc.metadata["section"])
    for doc in sections
]

expected_sections = list(
    range(1, 359)
)

print(
    "All sections valid:",
    section_numbers == expected_sections
)

print(
    "First 10:",
    section_numbers[:10]
)

print(
    "Last 10:",
    section_numbers[-10:]
)

All sections valid: True
First 10: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Last 10: [349, 350, 351, 352, 353, 354, 355, 356, 357, 358]


In [11]:
def show_section(
    section_number
):

    for doc in sections:

        if (
            doc.metadata["section"]
            == str(section_number)
        ):

            print(
                "LAW:",
                doc.metadata["law"]
            )

            print(
                "SECTION:",
                doc.metadata["section"]
            )

            print(
                "PAGES:",
                doc.metadata["pages"]
            )

            print("\nTEXT:\n")

            print(
                doc.page_content
            )

            return

In [12]:
show_section(21)

LAW: BNS
SECTION: 21
PAGES: []

TEXT:

21. Nothing is an offence which is done by a child above seven years of age and under
twelve years of age, who has not attained sufficient maturity of understanding to judge of
the nature and consequences of his conduct on that occasion.


In [13]:
show_section(4)

LAW: BNS
SECTION: 4
PAGES: []

TEXT:

4. The punishments to which offenders are liable under the provisions of this Sanhita
are—
(a) Death;
(b) Imprisonment for life;
(c) Imprisonment, which is of two descriptions, namely:—
(1) Rigorous, that is, with hard labour;
(2) Simple;
(d) Forfeiture of property;
(e) Fine;
(f) Community Service.


In [14]:
show_section(358)

LAW: BNS
SECTION: 358
PAGES: []

TEXT:

358. (1) The Indian Penal Code is herebyrepealed.
(2) Notwithstanding the repeal of the Code referred to in sub-section (1), it shall not
affect,—
(a) the previous operation of the Code so repealed or anything duly done or
suffered thereunder; or
(b) any right, privilege, obligation or liability acquired, accrued or incurred
under the Code so repealed; or
(c) any penalty, or punishment incurred in respect of any offences committed
against the Code so repealed; or
(d) anyinvestigation or remedyin respect of anysuch penalty, or punishment; or
(e) anyproceeding, investigation or remedy in respect of any such penalty or
punishment as aforesaid, and any such proceeding or remedy may be instituted,
continued or enforced, and any such penalty may be imposed as if that Code had not
been repealed.
(3) Notwithstanding such repeal, anything done or any action taken under the said
Code shall be deemed to have been done or taken under the corresponding provis

In [15]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(
    "Embedding dimension:",
    model.get_sentence_embedding_dimension()
)

print(
    "Maximum sequence length:",
    model.max_seq_length
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6830.90it/s]


Embedding dimension: 384
Maximum sequence length: 256


C:\Users\Himanshu\AppData\Local\Temp\ipykernel_11560\2645019028.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


In [16]:
splitter = (
    RecursiveCharacterTextSplitter
    .from_huggingface_tokenizer(

        tokenizer=model.tokenizer,

        chunk_size=180,

        chunk_overlap=25,

        separators=[
            "\n\n",
            "\n(",
            "\nExplanation",
            "\nIllustration",
            "\nIllustrations",
            "\nProvided",
            ". ",
            " "
        ]
    )
)

In [17]:
def create_chunks(
    sections
):

    chunks = []

    for section in sections:

        section_text = (
            section.page_content
        )

        # tokenize() avoids the >256 warning
        token_count = len(
            model.tokenizer.tokenize(
                section_text
            )
        )

        # --------------------------
        # Small section
        # --------------------------

        if token_count <= 180:

            chunks.append(
                Document(

                    page_content=
                        section_text,

                    metadata={
                        **section.metadata,

                        "parent_section":
                            section.metadata[
                                "section"
                            ],

                        "chunk_index": 0
                    }
                )
            )

        # --------------------------
        # Large section
        # --------------------------

        else:

            parts = splitter.split_text(
                section_text
            )

            for index, part in enumerate(
                parts
            ):

                chunks.append(
                    Document(

                        page_content=part,

                        metadata={
                            **section.metadata,

                            "parent_section":
                                section.metadata[
                                    "section"
                                ],

                            "chunk_index":
                                index
                        }
                    )
                )

    return chunks

In [18]:
chunks = create_chunks(
    sections
)

print(
    "Parent sections:",
    len(sections)
)

print(
    "Final child chunks:",
    len(chunks)
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (355 > 256). Running this sequence through the model will result in indexing errors


Parent sections: 358
Final child chunks: 683


In [19]:
for chunk in chunks:

    if (
        chunk.metadata["section"]
        == "21"
    ):

        print(
            "Chunk:",
            chunk.metadata[
                "chunk_index"
            ]
        )

        print(
            chunk.page_content
        )

        print("=" * 70)

Chunk: 0
21. Nothing is an offence which is done by a child above seven years of age and under
twelve years of age, who has not attained sufficient maturity of understanding to judge of
the nature and consequences of his conduct on that occasion.


In [20]:
def build_embedding_text(
    chunk
):

    return f"""Law: Bharatiya Nyaya Sanhita, 2023 (BNS)
Section: {chunk.metadata['section']}

{chunk.page_content}""".strip()

In [21]:
invalid_chunks = []

max_tokens = 0

for chunk in chunks:

    embedding_text = (
        build_embedding_text(
            chunk
        )
    )

    token_count = len(
        model.tokenizer.tokenize(
            embedding_text
        )
    )

    max_tokens = max(
        max_tokens,
        token_count
    )

    if (
        token_count
        > model.max_seq_length
    ):

        invalid_chunks.append({
            "section":
                chunk.metadata["section"],

            "chunk":
                chunk.metadata[
                    "chunk_index"
                ],

            "tokens":
                token_count
        })


print(
    "Maximum chunk tokens:",
    max_tokens
)

print(
    "Model maximum:",
    model.max_seq_length
)

print(
    "Invalid chunks:",
    len(invalid_chunks)
)

print(
    invalid_chunks[:10]
)

Maximum chunk tokens: 197
Model maximum: 256
Invalid chunks: 0
[]


In [22]:
embedding_texts = [

    build_embedding_text(
        chunk
    )

    for chunk in chunks
]

In [23]:
embeddings = model.encode(

    embedding_texts,

    batch_size=32,

    normalize_embeddings=True,

    show_progress_bar=True
)

Batches: 100%|██████████| 22/22 [00:14<00:00,  1.49it/s]


In [24]:
print(
    "Chunks:",
    len(chunks)
)

print(
    "Embeddings:",
    len(embeddings)
)

print(
    "Embedding shape:",
    embeddings.shape
)

Chunks: 683
Embeddings: 683
Embedding shape: (683, 384)


In [32]:
MONGO_URI = os.getenv(
    "MONGODB_URI"
)

client = MongoClient(
    MONGO_URI
)

db = client[
    "test_db"
]

collection = db[
    "test_collection"
]

print(
    client.admin.command(
        "ping"
    )
)

{'ok': 1}


In [33]:
mongo_documents = []

for i, chunk in enumerate(
    chunks
):

    section = (
        chunk.metadata[
            "section"
        ]
    )

    chunk_index = (
        chunk.metadata[
            "chunk_index"
        ]
    )

    mongo_documents.append({

        "id":
            f"bns-{section}-{chunk_index}",

        "values":
            embeddings[i].tolist(),

        "metadata": {

            "law":
                "BNS",

            "section":
                section,

            "parent_section":
                section,

            "chunk_index":
                chunk_index,

            "document_type":
                "primary_statute",

            "text":
                chunk.page_content,

            "pages":
                chunk.metadata[
                    "pages"
                ],

            "source":
                chunk.metadata[
                    "source"
                ]
        }
    })


print(
    "Prepared:",
    len(mongo_documents)
)

Prepared: 683


In [34]:
BATCH_SIZE = 500

for i in range(
    0,
    len(mongo_documents),
    BATCH_SIZE
):

    batch = mongo_documents[
        i:i + BATCH_SIZE
    ]

    collection.insert_many(
        batch
    )


print(
    "BNS records:",
    collection.count_documents(
        {
            "metadata.law":
                "BNS"
        }
    )
)

BNS records: 683


In [35]:
section_21 = collection.find_one(
    {
        "metadata.law":
            "BNS",

        "metadata.section":
            "21"
    }
)

print(
    section_21[
        "metadata"
    ]["text"]
)

21. Nothing is an offence which is done by a child above seven years of age and under
twelve years of age, who has not attained sufficient maturity of understanding to judge of
the nature and consequences of his conduct on that occasion.


In [47]:
def mongo_search(query,k=5):

    query_vector = model.encode([query],normalize_embeddings=True)[0].tolist()
    pipeline = [
        {
            "$vectorSearch": {

                "index":
                    "test_bns",

                "path":
                    "values",

                "queryVector":
                    query_vector,

                "numCandidates":
                    100,

                "limit":
                    k,

                "filter": {
                    "metadata.law":
                        "BNS"
                }
            }
        },

        {
            "$project": {

                "_id": 0,

                "id": 1,

                "metadata": 1,

                "score": {
                    "$meta":
                        "vectorSearchScore"
                }
            }
        }
    ]


    return list(
        collection.aggregate(
            pipeline
        )
    )

In [37]:
results = mongo_search("Can a 10 year old child be held criminally responsible?",k=5)

for result in results:

    print(
        "\nSECTION:",
        result[
            "metadata"
        ]["section"]
    )

    print(
        "SCORE:",
        round(
            result["score"],
            4
        )
    )

    print(
        result[
            "metadata"
        ]["text"][:500]
    )

In [38]:
from langchain_groq import ChatGroq


llm = ChatGroq(

    groq_api_key=
        os.getenv(
            "GROQ_API_KEY"
        ),

    model_name=
        "openai/gpt-oss-120b",

    temperature=0
)

In [39]:
def answer_question(
    question,
    k=5
):

    results = mongo_search(
        question,
        k=k
    )


    context = "\n\n".join(

        f"""
BNS Section {r['metadata']['section']}

{r['metadata']['text']}
""".strip()

        for r in results
    )


    prompt = f"""
You are a legal information assistant.

Answer the question using ONLY the provided
Bharatiya Nyaya Sanhita, 2023 context.

Rules:

1. Mention the relevant BNS section number.

2. Do not invent any section, exception,
punishment or legal provision.

3. Base the answer only on the supplied context.

4. If the context genuinely does not contain
the answer, say:
"Insufficient information in retrieved context."

CONTEXT:

{context}


QUESTION:

{question}


ANSWER:
"""


    response = llm.invoke(
        prompt
    )


    print(
        "RETRIEVED SECTIONS:"
    )

    for result in results:

        print(
            "Section",
            result[
                "metadata"
            ]["section"],

            "|",

            round(
                result["score"],
                4
            )
        )


    print(
        "\nANSWER:\n"
    )

    print(
        response.content
    )


    return {
        "question":
            question,

        "retrieved":
            results,

        "answer":
            response.content
    }

In [41]:
def extract_section_number(
    query
):

    match = re.search(

        r"\bsection\s+(\d{1,3})\b",

        query,

        re.IGNORECASE
    )

    if match:
        return match.group(1)

    return None

In [42]:
def retrieve_bns(
    query,
    k=5
):

    section_number = (
        extract_section_number(
            query
        )
    )

    # Exact lookup
    if section_number:

        documents = list(
            collection.find(
                {
                    "metadata.law":
                        "BNS",

                    "metadata.section":
                        section_number
                },

                {
                    "_id": 0
                }
            )
        )

        return documents

    # Semantic search
    return mongo_search(
        query,
        k=k
    )

In [51]:
answer_question(
    "What is organised crime under BNS Section 111?"
)

RETRIEVED SECTIONS:
Section 111 | 0.812
Section 190 | 0.7817
Section 111 | 0.7732
Section 112 | 0.7711
Section 111 | 0.7579

ANSWER:

**BNS Section 111**

Organised crime is defined in Section 111(1) as follows:

> “Any **continuing unlawful activity**—including kidnapping, robbery, vehicle theft, extortion, land‑grabbing, contract killing, economic offence, cyber‑crimes, trafficking of persons, drugs, weapons or illicit goods or services, human trafficking for prostitution or ransom—by any person or a group of persons acting in concert, singly or jointly, either as a **member of an organised crime syndicate** or on behalf of such syndicate, **by use of violence, threat of violence, intimidation, coercion, or by any other unlawful means** to obtain direct or indirect material benefit, including a financial benefit, shall constitute organised crime.”

The section further explains:

* **“Organised crime syndicate”** means a group of two or more persons who, acting either singly or jointl

{'question': 'What is organised crime under BNS Section 111?',
 'retrieved': [{'id': 'bns-111-0',
   'metadata': {'law': 'BNS',
    'section': '111',
    'parent_section': '111',
    'chunk_index': 0,
    'document_type': 'primary_statute',
    'text': '111. (1) Any continuing unlawful activity including kidnapping, robbery, vehicle\ntheft, extortion, land grabbing, contract killing, economic offence, cyber-crimes, trafficking\nof persons, drugs, weapons or illicit goods or services, human trafficking for prostitution or\nransom, by any person or a group of persons acting in concert, singly or jointly, either as a\nmember of an organised crime syndicate or on behalf of such syndicate, by use of violence,\nthreat of violence, intimidation, coercion, or by any other unlawful means to obtain direct or\nindirect material benefit including a financial benefit, shall constitute organised crime.\nExplanation.—For the purposes of this sub-section,––\n(i) “organised crime syndicate” means a gro